In [ ]:

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI
from pathlib import Path

# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")
    
# Step 1: Create your prompts

system_prompt = """
You are an HR Coordinator that analyzes the contents of a Curriculum Vitae,
and provides a short summary for the HR Manager, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt_prefix = """
Here are the contents CVs.
Stop analyzing the CVs when you have enough information to make a decision.
Provide a short summary of theses documents.
Give a score out of 10 for each CV. And our job requirements are as follows:
- 3 years of experience in software development
- Strong knowledge of Python
- Excellent communication skills
- Ability to work in a fast-paced environment
- Strong problem-solving skills
- Ability to work in a team
- Ability to learn new technologies quickly
- Ability to work independently
- Ability to handle multiple tasks
- Ability to handle stress
"""

# Step 2: Make the messages list
openai = OpenAI()

cvs = {}

for file in Path("day1Data").glob("*.txt"):
    with open(file, "r") as f:
        cvs[file.stem] = f.read()

messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + "\n".join(cvs.values())}
    ]

# Step 3: Call OpenAI

def summarize(msg):
    response = openai.chat.completions.create(
        model = "gpt-4.1-nano",
        messages = msg,
    )
    return response.choices[0].message.content

# Step 4: print the result
def display_summary(msg):
    summary = summarize(msg)
    display(Markdown(summary))


display_summary(messages)

**Candidate 1:**

- Experience: 4 years in software development
- Skills: Strong knowledge of Python, experience with multiple frameworks
- Communication: Good communication skills demonstrated through collaborative projects
- Work environment: Has worked in fast-paced environments
- Problem-solving: Proven problem-solving skills in previous roles
- Teamwork: Experienced in team settings
- Learning: Quick learner of new technologies
- Independent work: Comfortable working independently
- Multitasking & Stress: Demonstrated ability to manage multiple tasks and handle stress

**Score:** 9/10  
**Summary:** Meets all job requirements with extensive experience and strong skill set. Highly recommended.

---

**Candidate 2:**

- Experience: 2 years in software development
- Skills: Moderate Python knowledge, familiar with some frameworks
- Communication: Average communication skills
- Work environment: Limited exposure to fast-paced environments
- Problem-solving: Some problem-solving experience
- Teamwork: Some team experience, but limited
- Learning: Learning aptitude evident but less demonstrated
- Independent work: Some experience working independently
- Multitasking & Stress: Limited evidence of multitasking and stress management

**Score:** 6/10  
**Summary:** Candidate has potential but lacks sufficient experience and some required skills. Might be considered for a junior role or with additional training.